# Codification 1 : le mois suivant aide-t-il à retrouver la codification ?

Journal : [`tableau_resultats.md`](tableau_resultats.md). Essai **bonus et indépendant** des autres branches de machine learning ; `dpnm` n'est utilisé à aucun moment.

**Question** : la codification d'un mois M dépend-elle aussi de ce qui se passe **après** M ? Si les montants du mois suivant aident nettement à retrouver la codification de M, la banque la pose ou la corrige avec un temps de retard (paiement enregistré le mois suivant, codification modifiée après coup), ou sur un événement qu'elle voit plus tard : impossible alors de la deviner au moment M. Si le mois suivant n'apporte rien, la codification de M se décide sur le passé.

**Fenêtre** : pour retrouver la codification du mois M (`PAY_n`), on donne au modèle, selon le scénario, le plafond, la codification du mois précédent (`PAY_(n+1)`) et du mois suivant (`PAY_(n-1)`), les soldes et les paiements des mois précédent, M et suivant (`BILL_AMT(n+1)`, `BILL_AMTn`, `BILL_AMT(n-1)` ; `PAY_AMT(n+1)`, `PAY_AMTn`, `PAY_AMT(n-1)`). Les mois numérotés du plus récent au plus ancien : le mois suivant de M = n est n-1. Possible seulement pour mai, juin, juillet et août (`PAY_5` à `PAY_2`), seuls mois qui ont un mois avant et un mois après.

**Trois scénarios**, avec les mêmes modèles :

| Scénario | Variables |
|---|---|
| **S1, le passé seul** | plafond ; codification du mois précédent ; soldes et paiements du mois précédent et de M |
| **S2, plus les montants du mois suivant** | S1 + solde et paiement du mois suivant |
| **S3, plus la codification du mois suivant** | S2 + codification du mois suivant |

**Lecture** : un gain net de S2 sur S1 vient des **montants** du mois suivant, c'est le test principal. S3 gagnera de toute façon, par simple enchaînement (un client à 3 le mois suivant était en retard en M) : son gain ne prouve pas à lui seul une modification après coup, il faut regarder **où** il gagne (nouveaux retards, sorties du retard).

**Modèles** : CatBoost et le MLP, avec les meilleurs réglages du 01e, **sans boucle d'optimisation**, pour que la comparaison ne mesure que l'effet des variables ; montants et plafond au logarithme signé ; codifications regroupées (-2, -1, 0, 2 et plus). Points de comparaison : « même codification que le mois précédent » et « même codification que le mois suivant ».

**Données** : niveau 0 ; les 28 clients ayant une codification 1 d'avril à août sont retirés ; les exemples dont une codification de la fenêtre vaut 1 (septembre, pour la fenêtre d'août) sont retirés. Les clients sont coupés en 80/20 (`random_state=42`) : **les clients du test sont entièrement écartés**. Validation croisée à 5 plis **par client** (un client n'est jamais à la fois dans l'apprentissage et dans la validation d'un pli).

## 1. Données et exemples

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv").drop(columns="dpnm")   # dpnm jamais utilisé
PAY = [f"PAY_{n}" for n in range(1, 7)]
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
df = df[~anciens_1]
print(f"Clients retirés (codification 1 d'avril à août) : {int(anciens_1.sum())} ; clients restants : {len(df)}")

def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 et 1 inchangés
    return codifications.mask(codifications >= 2, 2)

# Clients : 80/20, le test est entièrement écarté
clients_train, clients_test = train_test_split(df.index, test_size=0.2, random_state=42, stratify=regrouper(df["PAY_2"]))

MOIS_NOMS = {2: "août", 3: "juillet", 4: "juin", 5: "mai"}
def exemples_du_mois(n):
    o = df.loc[clients_train]
    return pd.DataFrame({
        "LIMIT_BAL": o["LIMIT_BAL"],
        "codif_avant": regrouper(o[f"PAY_{n + 1}"]), "codif_apres": regrouper(o[f"PAY_{n - 1}"]),
        "solde_avant": o[f"BILL_AMT{n + 1}"], "solde_M": o[f"BILL_AMT{n}"], "solde_apres": o[f"BILL_AMT{n - 1}"],
        "paiement_avant": o[f"PAY_AMT{n + 1}"], "paiement_M": o[f"PAY_AMT{n}"], "paiement_apres": o[f"PAY_AMT{n - 1}"],
        "cible": regrouper(o[f"PAY_{n}"]), "mois": n, "client": o.index,
    })

emp = pd.concat([exemples_du_mois(n) for n in range(2, 6)], ignore_index=True)
avec_1 = (emp[["cible", "codif_avant", "codif_apres"]] == 1).any(axis=1)
emp = emp[~avec_1].reset_index(drop=True)
assert not emp["client"].isin(clients_test).any()
y = emp["cible"].astype(int)
print(f"Exemples retirés (une codification 1 dans la fenêtre) : {int(avec_1.sum())} ; exemples : {len(emp)} ({emp['client'].nunique()} clients)")
display(pd.crosstab(emp["mois"].map(MOIS_NOMS).rename("Mois M"), y.rename("Codification de M"), margins=True, margins_name="Total"))

# 5 plis par client
cv = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(emp, y, groups=emp["client"]))

Clients retirés (codification 1 d'avril à août) : 28 ; clients restants : 29972
Exemples retirés (une codification 1 dans la fenêtre) : 2937 ; exemples : 92971 (23977 clients)


Codification de M,-2,-1,0,2,Total
Mois M,,,,,
août,2038,4355,12581,2066,21040
juillet,3255,4752,12604,3366,23977
juin,3455,4569,13142,2811,23977
mai,3615,4425,13572,2365,23977
Total,12363,18101,51899,10608,92971


**Comment lire** : une ligne par mois M, une colonne par codification de M (regroupée). Les exemples de l'août comptent moins de clients : ceux codifiés 1 en septembre (le mois suivant) en sont retirés.

## 2. Les scénarios et les modèles

In [2]:
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.model_selection import cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SCENARIOS = {
    "S1 : passé seul": ["LIMIT_BAL", "codif_avant", "solde_avant", "solde_M", "paiement_avant", "paiement_M"],
    "S2 : + montants du mois suivant": ["LIMIT_BAL", "codif_avant", "solde_avant", "solde_M", "paiement_avant", "paiement_M",
                                         "solde_apres", "paiement_apres"],
    "S3 : + codification du mois suivant": ["LIMIT_BAL", "codif_avant", "solde_avant", "solde_M", "paiement_avant", "paiement_M",
                                             "solde_apres", "paiement_apres", "codif_apres"],
}
FAMILLES = [-2, -1, 0, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 2: "2 et plus"}

def famille_pay(codifications):
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def score_familles(vrai, predit):
    # Moyenne des taux de bonne famille (-2, -1, 0, 2 et plus), comme dans le 01e
    return balanced_accuracy_score(famille_pay(vrai), famille_pay(predit))

scoreur = make_scorer(score_familles)

def modeles(variables):
    categorielles = [v for v in variables if v.startswith("codif")]
    numeriques = [v for v in variables if not v.startswith("codif")]
    def log_montants(X):
        X = X.copy()
        X[numeriques] = np.sign(X[numeriques]) * np.log1p(np.abs(X[numeriques]))
        return X
    lineaire = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorielles),
                                  ("nombres", StandardScaler(), numeriques)])
    arbres = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorielles)],
                               remainder="passthrough")
    # Meilleurs réglages du 01e (journal, passage 9)
    return {
        "MLP": make_pipeline(FunctionTransformer(log_montants), lineaire,
                             MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-4, max_iter=500, early_stopping=True, random_state=42)),
        "CatBoost": make_pipeline(FunctionTransformer(log_montants), arbres,
                                  CatBoostClassifier(loss_function="MultiClass", iterations=1000, depth=5, learning_rate=0.03, l2_leaf_reg=5,
                                                     auto_class_weights="Balanced", random_seed=42, verbose=0, thread_count=-1,
                                                     allow_writing_files=False)),
    }

predictions = {}
for scenario, variables in SCENARIOS.items():
    for nom, modele in modeles(variables).items():
        predictions[(scenario, nom)] = np.asarray(cross_val_predict(modele, emp[variables], y, cv=cv, n_jobs=-1)).ravel().astype(int)
        print(f"{scenario}, {nom} : fait", flush=True)
predictions[("Repère", "Même codification que le mois précédent")] = emp["codif_avant"].to_numpy().astype(int)
predictions[("Repère", "Même codification que le mois suivant")] = emp["codif_apres"].to_numpy().astype(int)

S1 : passé seul, MLP : fait
S1 : passé seul, CatBoost : fait
S2 : + montants du mois suivant, MLP : fait
S2 : + montants du mois suivant, CatBoost : fait
S3 : + codification du mois suivant, MLP : fait
S3 : + codification du mois suivant, CatBoost : fait


**Comment lire** : chaque modèle prédit chaque exemple une fois, par le pli qui ne l'a pas appris (prédictions hors pli) ; les plis sont faits par client. Les deux repères ne sont pas des modèles : ils recopient la codification du mois précédent, ou celle du mois suivant.

## 3. Les scores des trois scénarios

In [3]:
lignes = []
for (scenario, nom), p in predictions.items():
    par_pli = [score_familles(y.iloc[val], p[val]) * 100 for _, val in cv]
    v, f = famille_pay(y), famille_pay(p)
    ligne = {"Scénario": scenario, "Modèle": nom,
             "Moyenne des 4 familles (%)": f"{np.mean(par_pli):.2f} ± {np.std(par_pli):.2f}",
             "Bonnes prédictions (%)": round((v == f).mean() * 100, 1)}
    ligne.update({f"Famille {NOMS_FAMILLES[c]} (%)": round((f[v == c] == c).mean() * 100, 1) for c in FAMILLES})
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index(["Scénario", "Modèle"]))

Moyenne des 4 familles (%)  \
Scénario                            Modèle                                                               
S1 : passé seul                     MLP                                                   81.41 ± 0.21   
                                    CatBoost                                              82.28 ± 0.33   
S2 : + montants du mois suivant     MLP                                                   91.31 ± 0.51   
                                    CatBoost                                              93.49 ± 0.26   
S3 : + codification du mois suivant MLP                                                   96.71 ± 0.22   
                                    CatBoost                                              96.83 ± 0.18   
Repère                              Même codification que le mois précédent               80.28 ± 0.25   
                                    Même codification que le mois suivant                 82.22 ± 0.24   

                                                                             Bonnes prédictions (%)  \
Scénario                            Modèle                                                            
S1 : passé seul                     MLP                                                        85.0   
                                    CatBoost                                                   84.3   
S2 : + montants du mois suivant     MLP                                                        93.8   
                                    CatBoost                                                   92.2   
S3 : + codification du mois suivant MLP                                                        97.4   
                                    CatBoost                                                   96.3   
Repère                              Même codification que le mois précédent                    84.0   
                                    Même codification que le mois suivant                      84.8   

                                                                             Famille -2 (%)  \
Scénario                            Modèle                                                    
S1 : passé seul                     MLP                                                93.7   
                                    CatBoost                                           96.2   
S2 : + montants du mois suivant     MLP                                                93.0   
                                    CatBoost                                           94.1   
S3 : + codification du mois suivant MLP                                                97.6   
                                    CatBoost                                           98.0   
Repère                              Même codification que le mois précédent            90.4   
                                    Même codification que le mois suivant              86.9   

                                                                             Famille -1 (%)  \
Scénario                            Modèle                                                    
S1 : passé seul                     MLP                                                77.7   
                                    CatBoost                                           79.2   
S2 : + montants du mois suivant     MLP                                                96.7   
                                    CatBoost                                           96.6   
S3 : + codification du mois suivant MLP                                                98.3   
                                    CatBoost                                           98.3   
Repère                              Même codification que le mois précédent            74.4   
                                    Même codification que le mois suivant              78.7   

                                                                             Famil

**Comment lire** : une ligne par scénario et par modèle ; la moyenne des 4 familles est donnée avec l'écart-type de ses 5 plis. **S2 contre S1** : ce qu'apportent les montants du mois suivant ; **S3 contre S2** : ce qu'apporte sa codification. Un écart plus petit que l'écart-type des plis n'est pas un gain.

## 4. Où les scénarios gagnent : les clients qui changent

Le même tableau que dans les autres notebooks, par rapport au mois précédent : sorties du retard (2 et plus le mois précédent, sain en M), retards maintenus, nouveaux retards (sain le mois précédent, 2 et plus en M), changements entre codifications saines.

In [4]:
famille_avant, famille_M = famille_pay(emp["codif_avant"]), famille_pay(y)
TYPES = {
    "Sortie du retard": (famille_avant == 2) & (famille_M != 2),
    "Retard maintenu (pas un changement)": (famille_avant == 2) & (famille_M == 2),
    "Nouveau retard": (famille_avant != 2) & (famille_M == 2),
    "Changement entre codifications saines": (famille_avant != 2) & (famille_M != 2) & (famille_avant != famille_M),
}
lignes = []
for type_changement, masque in TYPES.items():
    ligne = {"Type de changement": type_changement, "Exemples": int(masque.sum())}
    for (scenario, nom), p in predictions.items():
        ligne[f"{scenario.split(' :')[0]} {nom}" if scenario != "Repère" else nom] = round((famille_pay(p)[masque] == famille_M[masque]).mean() * 100, 1)
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index("Type de changement").T)

Type de changement,Sortie du retard,Retard maintenu (pas un changement),Nouveau retard,Changement entre codifications saines
Exemples,3030.0,7112.0,3496.0,8373.0
S1 MLP,18.8,96.3,0.0,20.3
S1 CatBoost,9.1,99.0,1.4,25.2
S2 MLP,69.1,86.6,65.2,81.1
S2 CatBoost,21.7,99.1,81.8,69.3
S3 MLP,95.9,97.6,83.6,90.3
S3 CatBoost,87.9,98.5,90.4,86.2
Même codification que le mois précédent,0.0,100.0,0.0,0.0
Même codification que le mois suivant,91.3,84.9,54.4,73.3


**Comment lire** : une ligne par scénario et modèle (ou repère), une colonne par type de changement ; la valeur est la part des exemples rangés dans leur vraie famille de M. Si S2 gagne surtout sur les nouveaux retards et les sorties du retard, ce sont les montants du mois suivant qui révèlent ces changements.

## 5. L'importance des variables (S2 et S3)

**Importance par permutation** (mêmes 5 plis, 5 mélanges par variable) : la baisse du score, en points, quand une variable est mélangée ; regroupée ensuite par position dans la fenêtre : mois précédent, mois M, mois suivant, plafond.

In [5]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance

def position(variable):
    return "plafond" if variable == "LIMIT_BAL" else {"avant": "mois précédent", "M": "mois M", "apres": "mois suivant"}[variable.rsplit("_", 1)[1]]

for scenario in ["S2 : + montants du mois suivant", "S3 : + codification du mois suivant"]:
    variables = SCENARIOS[scenario]
    importances = {}
    for nom, modele in modeles(variables).items():
        par_pli = []
        for apprentissage, validation in cv:
            m = clone(modele).fit(emp[variables].iloc[apprentissage], y.iloc[apprentissage])
            r = permutation_importance(m, emp[variables].iloc[validation], y.iloc[validation], scoring=scoreur, n_repeats=5, random_state=42, n_jobs=-1)
            par_pli.append(r.importances_mean * 100)
        importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=variables)
        print(f"{scenario}, {nom} : fait", flush=True)
    table = pd.DataFrame(importances)
    table["Moyenne des modèles"] = table.mean(axis=1)
    display(Markdown(f"**{scenario}, par variable**"))
    display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
    display(Markdown(f"**{scenario}, par position dans la fenêtre**"))
    display(table.groupby(table.index.map(position)).sum().reindex(["mois précédent", "mois M", "mois suivant", "plafond"]).round(2))

S2 : + montants du mois suivant, MLP : fait
S2 : + montants du mois suivant, CatBoost : fait


**S2 : + montants du mois suivant, par variable**

,MLP,CatBoost,Moyenne des modèles
codif_avant,38.29,31.75,35.02
paiement_apres,25.49,21.29,23.39
solde_M,25.46,15.62,20.54
solde_apres,14.94,13.79,14.36
solde_avant,4.23,1.33,2.78
paiement_M,0.94,0.32,0.63
LIMIT_BAL,0.73,0.16,0.45
paiement_avant,0.27,0.00,0.14


**S2 : + montants du mois suivant, par position dans la fenêtre**

,MLP,CatBoost,Moyenne des modèles
mois précédent,42.79,33.08,37.93
mois M,26.40,15.94,21.17
mois suivant,40.43,35.07,37.75
plafond,0.73,0.16,0.45


S3 : + codification du mois suivant, MLP : fait
S3 : + codification du mois suivant, CatBoost : fait


**S3 : + codification du mois suivant, par variable**

,MLP,CatBoost,Moyenne des modèles
codif_avant,25.88,20.66,23.27
paiement_apres,23.88,19.81,21.85
solde_M,21.85,15.31,18.58
codif_apres,16.46,11.68,14.07
solde_apres,4.45,4.01,4.23
solde_avant,2.17,0.63,1.40
paiement_M,0.78,0.12,0.45
paiement_avant,0.20,0.03,0.12
LIMIT_BAL,0.08,0.05,0.07


**S3 : + codification du mois suivant, par position dans la fenêtre**

,MLP,CatBoost,Moyenne des modèles
mois précédent,28.25,21.33,24.79
mois M,22.64,15.43,19.03
mois suivant,44.79,35.51,40.15
plafond,0.08,0.05,0.07


**Comment lire** : la valeur est la baisse du score, en points, quand la variable est mélangée. Dans S2, le poids des montants du mois suivant dit s'ils servent à retrouver la codification de M ; dans S3, celui de la codification du mois suivant, à lire avec la nuance de l'introduction (enchaînement naturel des retards).

## 5 bis. Recherche parallèle : la codification suit-elle le paiement du mois suivant, ou l'inverse ?

*Recherche parallèle, descriptive, sans modèle : elle ne fait pas partie de la correction de la codification 1 ; elle aide à lire le résultat des sections 3 à 5.*

Le paiement du mois suivant aide fortement à retrouver la codification de M (section 5). Deux lectures possibles, qui prédisent des comportements **opposés** :
1. **la codification de M porte sur le paiement du mois suivant** (décalage d'un mois) : les **nouveaux retards** de M **paient peu ou pas** le mois suivant, et c'est ce non-paiement qui fait poser le retard ; les **sorties du retard** paient le mois suivant ;
2. **le paiement du mois suivant réagit à la codification** : codifiés en retard en M, les clients sont relancés et **paient davantage** le mois suivant que les clients restés sains.

On compare donc, pour chaque type de passage entre le mois précédent et M, le paiement **du mois M** et celui **du mois suivant**, chacun rapporté à la facture qu'il règle (définition de `ratio_PAY_BILLn`, `docs/colonnes_creees.md` : paiement / facture exigible × 100, écrêté entre 0 et 200, 100 si aucune facture n'était exigible) : `PAY_AMTn` / `BILL_AMT(n+1)` pour le mois M, `PAY_AMT(n-1)` / `BILL_AMTn` pour le mois suivant. Calculé sur les exemples du train (mai à août).

In [6]:
def ratio(paiement, facture):
    return (paiement / facture.where(facture > 0) * 100).clip(lower=0, upper=200).fillna(100)

ratio_M = ratio(emp["paiement_M"], emp["solde_avant"])          # paiement de M / facture exigible en M
ratio_suivant = ratio(emp["paiement_apres"], emp["solde_M"])    # paiement du mois suivant / facture exigible le mois suivant
rien_M = (emp["paiement_M"] == 0) & (emp["solde_avant"] > 0)
rien_suivant = (emp["paiement_apres"] == 0) & (emp["solde_M"] > 0)

famille_avant, famille_M = famille_pay(emp["codif_avant"]), famille_pay(y)
GROUPES = {
    "Nouveau retard (sain avant, 2 et plus en M)": (famille_avant != 2) & (famille_M == 2),
    "Resté sain (sain avant et en M)": (famille_avant != 2) & (famille_M != 2),
    "Sortie du retard (2 et plus avant, sain en M)": (famille_avant == 2) & (famille_M != 2),
    "Retard maintenu (2 et plus avant et en M)": (famille_avant == 2) & (famille_M == 2),
}
lignes = []
for groupe, masque in GROUPES.items():
    lignes.append({"Passage": groupe, "Exemples": int(masque.sum()),
                   "Ratio de paiement de M — médiane (%)": round(ratio_M[masque].median(), 1),
                   "Ratio de paiement du mois suivant — médiane (%)": round(ratio_suivant[masque].median(), 1),
                   "Rien payé en M, facture due (%)": round(rien_M[masque].mean() * 100, 1),
                   "Rien payé le mois suivant, facture due (%)": round(rien_suivant[masque].mean() * 100, 1),
                   "Facture soldée le mois suivant (ratio >= 100 %) (%)": round((ratio_suivant[masque] >= 100).mean() * 100, 1)})
display(pd.DataFrame(lignes).set_index("Passage"))

,Exemples,Ratio de paiement de M — médiane (%),Ratio de paiement du mois suivant — médiane (%),"Rien payé en M, facture due (%)","Rien payé le mois suivant, facture due (%)",Facture soldée le mois suivant (ratio >= 100 %) (%)
Passage,,,,,,
"Nouveau retard (sain avant, 2 et plus en M)",3496,5.4,0.0,4.5,72.9,8.3
Resté sain (sain avant et en M),79333,13.0,12.3,1.9,1.5,38.4
"Sortie du retard (2 et plus avant, sain en M)",3030,0.0,13.4,68.3,0.1,19.5
Retard maintenu (2 et plus avant et en M),7112,3.4,3.8,43.7,36.5,0.0


**Comment lire** : une ligne par type de passage, avec le paiement du mois M et celui du mois suivant. **Lecture 1** si les nouveaux retards paient nettement **moins** le mois suivant que les clients restés sains (beaucoup de « rien payé le mois suivant »), et les sorties du retard nettement **plus** ; **lecture 2** si les nouveaux retards paient **plus** le mois suivant que les restés sains. Les deux mécanismes peuvent coexister : le tableau dit lequel domine.

## 6. Conclusion

*À écrire après lecture des résultats.*